In [1]:
# https://www.kaggle.com/dgomonov/new-york-city-airbnb-open-data

In [ ]:
# An example work on the same data set 
# https://www.kaggle.com/wguesdon/nyc-airbnb-eda-visualization-regression#5.-Modeling-second-model

In [2]:
! ls -l /home/train/datasets | grep NYC

-rw-rw-r--. 1 train train  7077973 Feb 28 14:33 AB_NYC_2019.csv


In [3]:
import findspark

findspark.init("/opt/manual/spark")

from pyspark.sql import SparkSession, functions as F
import pandas as pd

spark = SparkSession.builder \
.appName("House Price Regression") \
.master("local[2]") \
.config("spark.executer.memory","3g") \
.config("spark.sql.shuffle.partitions", "1") \
.getOrCreate()

## Read data

In [225]:
df = spark.read.format("csv") \
.option("header",True) \
.option("inferSchema", True) \
.load("file:///home/train/datasets/AB_NYC_2019.csv")

In [226]:
df.cache()

DataFrame[id: string, name: string, host_id: string, host_name: string, neighbourhood_group: string, neighbourhood: string, latitude: string, longitude: string, room_type: string, price: string, minimum_nights: string, number_of_reviews: string, last_review: string, reviews_per_month: string, calculated_host_listings_count: string, availability_365: int]

In [227]:
print(len(df.columns))

16


## Explore the data

In [228]:
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

In [229]:
df.limit(5).toPandas()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.9419,Private room,150,3,0,None,None,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [230]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- host_id: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- neighbourhood_group: string (nullable = true)
 |-- neighbourhood: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- price: string (nullable = true)
 |-- minimum_nights: string (nullable = true)
 |-- number_of_reviews: string (nullable = true)
 |-- last_review: string (nullable = true)
 |-- reviews_per_month: string (nullable = true)
 |-- calculated_host_listings_count: string (nullable = true)
 |-- availability_365: integer (nullable = true)



In [231]:
# Discart: id, name, 
# host_id must be integer
# latitude and longitude must be float
# price must be integer
# minimum_nights must be integer
# number_of_reviews must be integer
# last_review must be date
# reviews_per_month must be float

In [232]:
my_schema = "id string, \
name string, \
host_id int, \
host_name string, \
neighbourhood_group string, \
neighbourhood string, \
latitude float, \
longitude float, \
room_type string, \
price int, \
minimum_nights int, \
number_of_reviews int, \
last_review date, \
reviews_per_month float, \
calculated_host_listings_count int, \
availability_365 int " 

In [233]:
df = spark.read.format("csv") \
.option("header",True) \
.schema(my_schema) \
.load("file:///home/train/datasets/AB_NYC_2019.csv")

In [234]:
df.limit(5).toPandas()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.647491,-73.972366,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.753620,-73.983772,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.809021,-73.941902,Private room,150,3,0,None,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.685139,-73.959763,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.798512,-73.943993,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [235]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- host_id: integer (nullable = true)
 |-- host_name: string (nullable = true)
 |-- neighbourhood_group: string (nullable = true)
 |-- neighbourhood: string (nullable = true)
 |-- latitude: float (nullable = true)
 |-- longitude: float (nullable = true)
 |-- room_type: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- minimum_nights: integer (nullable = true)
 |-- number_of_reviews: integer (nullable = true)
 |-- last_review: date (nullable = true)
 |-- reviews_per_month: float (nullable = true)
 |-- calculated_host_listings_count: integer (nullable = true)
 |-- availability_365: integer (nullable = true)



In [236]:
df.count()

49079

## Null check

In [237]:
avg_dict = {}

In [238]:
df_count = df.count()

In [239]:
# Seperate categorical and numeric null including columns
cat_nulls = []

num_nulls = []

In [240]:
def null_count(df, col_name):
    nc = df.filter( 
        (F.col(col_name).isNull()) | 
        (F.col(col_name) == "") | 
        (F.col(col_name) == "NA")
    ).count()
    
    return nc

In [241]:
for col_name in df.dtypes:
    
    nc = null_count(df, col_name[0])
    
    
    if(  nc > 0 ):
        print("{} {} type has {} null values, % {}".format(col_name[0], col_name[1], nc, (nc/df_count)))
        if col_name[1] == 'string':
            cat_nulls.append(col_name[0])
        else:
            num_nulls.append(col_name[0])

name string type has 32 null values, % 0.0006520100246541291
host_id int type has 350 null values, % 0.007131359644654537
host_name string type has 206 null values, % 0.004197314533710956
neighbourhood_group string type has 185 null values, % 0.0037694329550316836
neighbourhood string type has 185 null values, % 0.0037694329550316836
latitude float type has 194 null values, % 0.003952810774465658
longitude float type has 343 null values, % 0.006988732451761446
room_type string type has 185 null values, % 0.0037694329550316836
price int type has 194 null values, % 0.003952810774465658
minimum_nights int type has 188 null values, % 0.0038305588948430083
number_of_reviews int type has 342 null values, % 0.006968357138491004
last_review date type has 10379 null values, % 0.21147537643391268
reviews_per_month float type has 10221 null values, % 0.2082560769371829
calculated_host_listings_count int type has 192 null values, % 0.003912060147924774
availability_365 int type has 342 null values

In [242]:
print(cat_nulls)

['name', 'host_name', 'neighbourhood_group', 'neighbourhood', 'room_type']


In [243]:
print(num_nulls)

['host_id', 'latitude', 'longitude', 'price', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365']


# Explore Categoricals

In [244]:
for col_name in df.dtypes:
    if col_name[1] == 'string':
        print(df.select(col_name[0]).distinct().count())
        df.groupBy(col_name[0]).count().orderBy(F.desc("count")).show()

49066
+--------------------+-----+
|                  id|count|
+--------------------+-----+
|NEAR TO YANKE STA...|    5|
|   Se habla Español"|    3|
|15 min F Times Sq...|    3|
|        Mount Sinai"|    2|
|    muy cercadetodo"|    2|
|         Brownstone"|    2|
|               SoBro|    2|
|   AFFORDABLE PRICE"|    2|
|                2539|    1|
|                2595|    1|
|                3831|    1|
|                5178|    1|
|                3647|    1|
|                5022|    1|
|                5099|    1|
|                5203|    1|
|                5238|    1|
|                5441|    1|
|                5121|    1|
|                5295|    1|
+--------------------+-----+
only showing top 20 rows

48041
+--------------------+-----+
|                name|count|
+--------------------+-----+
|                null|   32|
|      Hillside Hotel|   18|
| Home away from home|   18|
|New york Multi-un...|   16|
|  Brooklyn Apartment|   12|
|        Private Room|   11|
|Loft

In [245]:
# Discard: id, name, host_name, neighbourhood
# Group waek categories in neighbourhood_group lt 200
# Group waek categories in room_type lt 200

# Group weak classes in room_type and neighbourhood_group

In [246]:
def find_weaks(df, col_name, limit):
    '''
    Finds weak classes in a given columns according to a limit.
    For example: If I consider frequency of a class less than 200 I can consider it is waek.
    Function returns a list including weak classes
    '''
    below_limits = df.groupBy(col_name).count() \
    .filter(F.col("count") < limit) \
    .select(col_name).collect()
    to_be_grouped = [i[0] for i in below_limits]
    return to_be_grouped

In [247]:
room_type_to_be_grouped = find_weaks(df, "room_type", 200)

In [248]:
print(room_type_to_be_grouped)

['89', '99', '225', '56', '69', '170', '-73.95777', '298', '59', '50', '80', '65', '-73.95179', '160', '-73.94639', '140', '200', '45', '67', '86', '72', '95', '230', '110', 'Howard', '250', '119', '138', '350', '150', '93', '188', '60', '40', '333', '63', '100', '88', '300', '425', '175', '159', '-73.90783', '55', '79', '31', '75', '120', '116', '410', '135', '40.71509', '70', '46', '128', '38', '81', '35', '54', '109', '145', '179', '185', '6500', '42', '105', '130', '180', None, '205', '85', '39', '-73.94861', '279', '57', '51', '40.57453', '299', '78', '48', '-74.00244', '249', '68', '115']


In [249]:
neighbourhood_group_to_be_grouped = find_weaks(df, "neighbourhood_group", 200)

In [250]:
print(neighbourhood_group_to_be_grouped)

['Williamsburg', 'East Harlem', 'SoHo', 'Eltingville', 'Gramercy', 'Astoria', 'William Hakan', 'Midwood', 'Harlem', 'Longwood', 'Seth', 'Fort Greene', 'Prospect-Lefferts Gardens', 'Nadia', 'East Village', 'Elmhurst', 'Throgs Neck', 'Morningside Heights', 'Bushwick', 'Clinton Hill', 'Upper West Side', 'Springfield Gardens', 'Woodhaven', 'Ditmars Steinway', 'D', 'Arverne', 'Bath Beach', 'Murray Hill', 'East New York', 'Long Island City', 'Flushing', 'Inwood', 'Mott Haven', 'Midtown', 'Douglaston', 'Little Neck', 'Greenwich Village', 'NoHo', 'Crown Heights', "Hell's Kitchen", 'Woodside', 'Bedford-Stuyvesant', 'Carmen', 'Canarsie', 'Upper East Side', 'Randall Manor', 'Washington Heights', 'East Flatbush', 'South Ozone Park', '197400421', 'Prospect Heights', 'Borough Park', 'Red Hook', 'Chelsea', 'Greenpoint', 'Flatbush', 'East Elmhurst', 'Jackson Heights', None, 'Queens Village', 'Edgemere', 'Maspeth', 'Krista', 'Stuyvesant Town', 'West Village', '194716858', 'Cambria Heights', 'Pelham Gar

In [251]:
# If null or any other weak class impute as Other
df2 = df.withColumn("room_type", 
                    F.when( F.col("room_type").isin(room_type_to_be_grouped) | 
                           F.col("room_type").isNull(), 
                          "Other").otherwise(F.col("room_type"))) \
.withColumn("neighbourhood_group", 
                    F.when( F.col("neighbourhood_group").isin(neighbourhood_group_to_be_grouped) | 
                          F.col("neighbourhood_group").isNull(), 
                          "Other").otherwise(F.col("neighbourhood_group")))

In [252]:
df2.groupBy("room_type").count().orderBy(F.desc("count")).show()

+---------------+-----+
|      room_type|count|
+---------------+-----+
|Entire home/apt|25345|
|   Private room|22226|
|    Shared room| 1158|
|          Other|  350|
+---------------+-----+



In [253]:
df2.groupBy("neighbourhood_group").count().orderBy(F.desc("count")).show()

+-------------------+-----+
|neighbourhood_group|count|
+-------------------+-----+
|          Manhattan|21594|
|           Brooklyn|20055|
|             Queens| 5630|
|              Bronx| 1080|
|      Staten Island|  370|
|              Other|  350|
+-------------------+-----+




# Group columns

In [254]:
categoric_cols = []

numeric_cols = []

discarted_cols = ['id', 'name', 'host_id', 'host_name', 'neighbourhood']


label_col = ['price']

In [255]:
# column seperation
for col_name in df.dtypes:
    if (col_name[0] not in label_col+discarted_cols):
        if (col_name[1] == 'string'):
            categoric_cols.append(col_name[0])
        else: numeric_cols.append(col_name[0])

In [256]:
print(categoric_cols)
print(len(categoric_cols))

['neighbourhood_group', 'room_type']
2


In [257]:
print(numeric_cols)
print(len(numeric_cols))

['latitude', 'longitude', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365']
8


In [258]:
print(discarted_cols)
print(len(discarted_cols))

['id', 'name', 'host_id', 'host_name', 'neighbourhood']
5


In [259]:
print(label_col)
print(len(label_col))

['price']
1


### column check

In [260]:
len(df.columns) == (len(categoric_cols)  + len(numeric_cols) + len(discarted_cols) + len(label_col))

True

# Dealing with nulls

In [261]:
cat_nulls = []
num_nulls = []
for col_name in df2.dtypes:
    if col_name[0] not in discarted_cols:
        nc = null_count(df2, col_name[0])
    
    
        if(  nc > 0 ):
            print("{} {} type has {} null values, % {}".format(col_name[0], col_name[1], nc, (nc/df_count)))
            if col_name[1] == 'string':
                cat_nulls.append(col_name[0])
            else:
                num_nulls.append(col_name[0])

latitude float type has 194 null values, % 0.003952810774465658
longitude float type has 343 null values, % 0.006988732451761446
price int type has 194 null values, % 0.003952810774465658
minimum_nights int type has 188 null values, % 0.0038305588948430083
number_of_reviews int type has 342 null values, % 0.006968357138491004
last_review date type has 10379 null values, % 0.21147537643391268
reviews_per_month float type has 10221 null values, % 0.2082560769371829
calculated_host_listings_count int type has 192 null values, % 0.003912060147924774
availability_365 int type has 342 null values, % 0.006968357138491004


In [262]:
print(cat_nulls)

[]


In [263]:
print(num_nulls)

['latitude', 'longitude', 'price', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365']


# latitude Nulls

In [264]:
df2.filter("latitude IS NULL").limit(10).toPandas()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,255476,The BLUE OWL:,NaN,None,Other,None,NaN,NaN,Other,NaN,NaN,NaN,None,NaN,NaN,NaN
1,267708,Charming Hotel Alternative,NaN,None,Other,None,NaN,NaN,Other,NaN,NaN,NaN,None,NaN,NaN,NaN
2,405408,Magazine SOHO Studio Loft.,NaN,None,Other,None,NaN,NaN,Other,NaN,NaN,NaN,None,NaN,NaN,NaN
3,1096326,Private studio in staten island,NaN,None,Other,None,NaN,NaN,Other,NaN,NaN,NaN,None,NaN,NaN,NaN
4,",6024006,Johnny,Staten Island,St. George,40.64...",None,NaN,None,Other,None,NaN,NaN,Other,NaN,NaN,NaN,None,NaN,NaN,NaN
5,3465651,“No Place Like Home”,NaN,None,Other,None,NaN,NaN,Other,NaN,NaN,NaN,None,NaN,NaN,NaN
6,3757461,Manhattan Sun Drenched,NaN,None,Other,None,NaN,NaN,Other,NaN,NaN,NaN,None,NaN,NaN,NaN
7,3768977,2 bedroom Duplex condo,NaN,None,Other,None,NaN,NaN,Other,NaN,NaN,NaN,None,NaN,NaN,NaN
8,4681219,"""""""Central Park"""" View Prvt Room",NaN,24208781,Other,Manhattan,NaN,40.801262,Other,NaN,70.0,2.0,None,NaN,NaN,1.0
9,4756578,Clean | Green | Spacious,NaN,None,Other,None,NaN,NaN,Other,NaN,NaN,NaN,None,NaN,NaN,NaN


In [265]:
# It seems nulls are in the same rows

In [266]:
len(df.columns)

16

In [267]:
df3 = df2.dropna(thresh=7, subset=num_nulls)

In [268]:
print("df3 count", df3.count())
print("df2 count", df2.count())

df3 count 48729
df2 count 49079


In [269]:
cat_nulls = []
num_nulls = []
for col_name in df3.dtypes:
    if col_name[0] not in discarted_cols:
        nc = null_count(df3, col_name[0])
    
    
        if(  nc > 0 ):
            print("{} {} type has {} null values, % {}".format(col_name[0], col_name[1], nc, (nc/df_count)))
            if col_name[1] == 'string':
                cat_nulls.append(col_name[0])
            else:
                num_nulls.append(col_name[0])

last_review date type has 10029 null values, % 0.20434401678925815
reviews_per_month float type has 10029 null values, % 0.20434401678925815


In [341]:
df4 = df3.withColumn("reviews_per_month", F.when( F.col("number_of_reviews") == 0, 
                          0.0).otherwise(F.col("reviews_per_month")))

In [342]:
for col_name in df4.dtypes:
    if col_name[0] not in discarted_cols:
        nc = null_count(df4, col_name[0])
    
    
        if(  nc > 0 ):
            print("{} {} type has {} null values, % {}".format(col_name[0], col_name[1], nc, (nc/df_count)))

In [343]:
df4.where("number_of_reviews = 0").limit(7).toPandas()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.809021,-73.941902,Private room,150,3,0,None,0.0,1,365
1,7750,Huge 2 BR Upper East Cental Park,17985,Sing,Manhattan,East Harlem,40.796848,-73.948723,Entire home/apt,190,7,0,None,0.0,2,249
2,8700,Magnifique Suite au N de Manhattan - vue Cloitres,26394,Claude & Sophie,Manhattan,Inwood,40.867538,-73.926392,Private room,80,4,0,None,0.0,1,0
3,11452,Clean and Quiet in Brooklyn,7355,Vt,Brooklyn,Bedford-Stuyvesant,40.688759,-73.943123,Private room,35,60,0,None,0.0,1,365
4,11943,Country space in the city,45445,Harriet,Brooklyn,Flatbush,40.637020,-73.963272,Private room,150,1,0,None,0.0,1,365
5,51438,1 Bedroom in 2 Bdrm Apt- Upper East,236421,Jessica,Manhattan,Upper East Side,40.773331,-73.951988,Private room,130,14,0,None,0.0,2,0
6,54466,Beautiful Uptown Manhattan apartmnt,253385,Douglas,Manhattan,Harlem,40.802341,-73.956032,Private room,200,30,0,None,0.0,1,365


# Exclude price = 0

In [375]:
df5 = df4.filter("price > 0")

In [376]:
print("df2 count", df2.count())
print("df3 count", df3.count())
print("df4 count", df4.count())
print("df5 count", df5.count())

df2 count 49079
df3 count 48729
df4 count 48729
df5 count 48718


In [377]:
# column seperation
categoric_cols = []

numeric_cols = []

discarted_cols = ['id', 'name', 'host_id', 'host_name', 'neighbourhood', 'last_review']


label_col = ['price']

for col_name in df5.dtypes:
    if (col_name[0] not in label_col+discarted_cols):
        if (col_name[1] == 'string'):
            categoric_cols.append(col_name[0])
        else: numeric_cols.append(col_name[0])

In [378]:
df5.select(categoric_cols+numeric_cols+label_col).limit(6).toPandas()

,neighbourhood_group,room_type,latitude,longitude,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,price
0,Brooklyn,Private room,40.647491,-73.972366,1,9,0.21,6,365,149
1,Manhattan,Entire home/apt,40.753620,-73.983772,1,45,0.38,2,355,225
2,Manhattan,Private room,40.809021,-73.941902,3,0,0.00,1,365,150
3,Brooklyn,Entire home/apt,40.685139,-73.959763,1,270,4.64,1,194,89
4,Manhattan,Entire home/apt,40.798512,-73.943993,10,9,0.10,1,0,80
5,Manhattan,Entire home/apt,40.747669,-73.974998,3,74,0.59,1,129,200


In [380]:
last_df = df5

## Which columns will be hot coded?

In [346]:
one_hot_cols = []

In [381]:
for col_name in categoric_cols:
    dist_count = last_df.select(col_name).distinct().count()
    print(col_name, dist_count)
    if  dist_count > 2:
        one_hot_cols.append(col_name)

neighbourhood_group 5
room_type 3


In [382]:
print(one_hot_cols)

['neighbourhood_group', 'room_type', 'neighbourhood_group', 'room_type', 'neighbourhood_group', 'room_type']


# StringIndexer

In [383]:
from pyspark.ml.feature import StringIndexer

In [384]:
my_dict = {}

string_indexer_objs = []

string_indexer_output_names = []

ohe_input_names = []

ohe_output_names = []

for col_name in categoric_cols:
    my_dict[col_name+"_index_obj"] = StringIndexer() \
                                    .setHandleInvalid("skip") \
                                    .setInputCol(col_name) \
                                    .setOutputCol(col_name+"_indexed")
    
    string_indexer_objs.append(my_dict.get(col_name+"_index_obj"))
    string_indexer_output_names.append(col_name+"_indexed")
    if col_name in one_hot_cols:
        ohe_input_names.append(col_name+"_indexed")
        ohe_output_names.append(col_name+"_ohe")

In [385]:
print(string_indexer_objs)
print(len(string_indexer_objs))

[StringIndexer_08117c5b5760, StringIndexer_cb7a78512f50]
2


In [386]:
print(string_indexer_output_names)
print(len(string_indexer_output_names))

['neighbourhood_group_indexed', 'room_type_indexed']
2


In [387]:
print(ohe_input_names)
print(len(ohe_input_names))

['neighbourhood_group_indexed', 'room_type_indexed']
2


In [388]:
print(ohe_output_names)
print(len(ohe_output_names))

['neighbourhood_group_ohe', 'room_type_ohe']
2


# OnehotEncoder

In [389]:
from pyspark.ml.feature import OneHotEncoder

In [390]:
encoder = OneHotEncoder() \
.setInputCols(ohe_input_names) \
.setOutputCols(ohe_output_names)

# VectorAssembler

In [391]:
from pyspark.ml.feature import VectorAssembler

In [392]:
not_to_hot_coded = list(set(string_indexer_output_names).difference(set(ohe_input_names)))

print(not_to_hot_coded)

[]


In [393]:
if len(categoric_cols) == (len(not_to_hot_coded)+len(ohe_output_names)):
    print("Columns verified")
else: print("Columns are not verified")

Columns verified


In [394]:
assembler = VectorAssembler().setHandleInvalid("skip") \
.setInputCols(numeric_cols+not_to_hot_coded+ohe_output_names) \
.setOutputCol('unscaled_features')

# Feature Scaling

In [411]:
from pyspark.ml.feature import StandardScaler

In [412]:
scaler = StandardScaler().setInputCol("unscaled_features").setOutputCol("scaled_features")

# PCA

In [413]:
from pyspark.ml.feature import PCA

In [427]:
pca = PCA().setInputCol("unscaled_features").setK(5).setOutputCol("features")

# Estimator

In [441]:
from pyspark.ml.regression import RandomForestRegressor

In [442]:
estimator = RandomForestRegressor(numTrees=400) \
.setFeaturesCol("unscaled_features") \
.setLabelCol(label_col[0])

# Pipeline

In [443]:
pipeline_obj = Pipeline().setStages(string_indexer_objs + [encoder, assembler, estimator])

In [444]:
from pyspark.ml import Pipeline

# Split test train

In [445]:
train_df, test_df = last_df.randomSplit([.2, .8], seed=142)

# Train the model

In [446]:
pipeline_model = pipeline_obj.fit(train_df)

# Prediction

In [447]:
transformed_df = pipeline_model.transform(test_df)

In [448]:
transformed_df.select(label_col[0],'prediction').show(truncate=False)

+-----+------------------+
|price|prediction        |
+-----+------------------+
|67   |78.0303815903408  |
|57   |68.45991093190594 |
|199  |233.93804375183922|
|105  |170.60871518692016|
|485  |251.2167470203853 |
|120  |157.5252069721527 |
|125  |173.78726615422963|
|150  |172.32363089837094|
|95   |123.74568836990119|
|200  |160.45002782406854|
|90   |245.53155206128613|
|100  |173.52811918276348|
|50   |67.03384344362513 |
|70   |66.93293202896628 |
|110  |253.94725925965113|
|50   |83.7983563553404  |
|175  |172.17784612900482|
|70   |124.43580866263898|
|51   |75.42022509281607 |
|80   |151.49287346100834|
+-----+------------------+
only showing top 20 rows



# Evaluate the model

In [449]:
from pyspark.ml.evaluation import RegressionEvaluator

In [450]:
evaluator = RegressionEvaluator(labelCol=label_col[0], metricName='r2')

In [451]:
evaluator.evaluate(transformed_df)

0.0909445201488982

In [452]:
evaluator.getMetricName()

'r2'

In [80]:
spark.stop()